# QuantMind — EDA (Exploratory Data Analysis)

Analyse exploratoire des données financières utilisées par QuantMind.

**Checkpoints couverts :**
1. Visualiser les prix de clôture
2. Identifier les valeurs manquantes
3. Analyser les volumes
4. Générer la matrice de corrélation
5. Visualiser les returns journaliers
6. Documenter les observations

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style='darkgrid', palette='deep')
    SNS = True
except ImportError:
    SNS = False

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True

DATA_DIR = 'data/processed'
TICKERS  = ['AAPL']

def load(ticker):
    path = os.path.join(DATA_DIR, f"{ticker.replace('-','_')}.csv")
    if not os.path.exists(path):
        return None
    return pd.read_csv(path, index_col='Date', parse_dates=True)

data = {t: load(t) for t in TICKERS}
data = {t: d for t, d in data.items() if d is not None}
print(f'Tickers chargés : {list(data.keys())}')
for t, d in data.items(): 
    print(f'  {t:8s} : {len(d):4d} jours  | {d.index.min().date()} → {d.index.max().date()}')

## 1. Aperçu des données

In [ ]:
ticker = next(iter(data))
df = data[ticker]
print(f'--- {ticker} ---')
print(df.info())
df.describe().T[['mean','std','min','max']]

## 2. Valeurs manquantes

In [ ]:
missing = pd.DataFrame({t: d.isna().sum() for t, d in data.items()})
missing = missing.loc[missing.sum(axis=1) > 0]
if missing.empty:
    print('✓ Aucune valeur manquante détectée (dropna() appliqué dans compute_indicators)')
else:
    print(missing)

gaps = []
for t, d in data.items():
    diff = d.index.to_series().diff().dt.days
    big_gaps = (diff > 5).sum()
    gaps.append({'ticker': t, 'jours': len(d), 'gaps_>5j': int(big_gaps)})
pd.DataFrame(gaps)

## 3. Prix de clôture (normalisé base 100)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for t, d in data.items():
    norm = 100 * d['Close'] / d['Close'].iloc[0]
    ax.plot(d.index, norm, label=t, linewidth=1.2)
ax.set_title('Performance relative — base 100')
ax.set_ylabel('Indice (base 100)')
ax.legend(loc='best', ncol=3)
plt.tight_layout(); plt.show()

In [ ]:
n = len(data)
fig, axes = plt.subplots(n, 1, figsize=(14, 2.5*n), sharex=False)
if n == 1: axes = [axes]
for ax, (t, d) in zip(axes, data.items()):
    ax.plot(d.index, d['Close'], color='#00d68f', linewidth=1)
    if 'EMA_20' in d: ax.plot(d.index, d['EMA_20'], color='#3b82f6', linewidth=0.8, alpha=0.7, label='EMA 20')
    if 'EMA_50' in d: ax.plot(d.index, d['EMA_50'], color='#f59e0b', linewidth=0.8, alpha=0.7, label='EMA 50')
    ax.set_title(f'{t} — Prix de clôture')
    ax.set_ylabel('USD')
    ax.legend(loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

## 4. Volumes

In [ ]:
fig, axes = plt.subplots(len(data), 1, figsize=(14, 2*len(data)), sharex=False)
if len(data) == 1: axes = [axes]
for ax, (t, d) in zip(axes, data.items()):
    ax.bar(d.index, d['Volume'], color='#3b82f6', alpha=0.6, width=1.0)
    ax.set_title(f'{t} — Volume journalier')
    ax.set_yscale('log')
plt.tight_layout(); plt.show()

vol_stats = pd.DataFrame({
    t: {
        'vol_moyen': d['Volume'].mean(),
        'vol_max':   d['Volume'].max(),
        'jours_pic': (d['Volume'] > d['Volume'].mean() * 3).sum(),
    } for t, d in data.items()
}).T
vol_stats

## 5. Returns journaliers

In [ ]:
returns = pd.DataFrame({t: d['Close'].pct_change() for t, d in data.items()}).dropna()
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, t in zip(axes.flat, returns.columns):
    ax.hist(returns[t]*100, bins=60, color='#00d68f', alpha=0.7, edgecolor='#0d1117')
    ax.axvline(0, color='#f43f5e', linewidth=1, linestyle='--')
    ax.set_title(f'{t} — Returns (%)')
    ax.set_xlabel('Return %'); ax.set_ylabel('Fréquence')
plt.tight_layout(); plt.show()

stats = returns.agg(['mean', 'std', 'min', 'max', 'skew', 'kurt']).T
stats['mean_pct']   = stats['mean'] * 100
stats['std_pct']    = stats['std'] * 100
stats['ann_vol_%']  = stats['std'] * np.sqrt(252) * 100
stats['sharpe_proxy'] = (stats['mean'] / (stats['std']+1e-10)) * np.sqrt(252)
stats[['mean_pct','std_pct','ann_vol_%','sharpe_proxy','skew','kurt']].round(3)

## 6. Matrice de corrélation

In [ ]:
corr = returns.corr()
fig, ax = plt.subplots(figsize=(7, 6))
if SNS:
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax,
                vmin=-1, vmax=1, square=True, linewidths=0.5)
else:
    im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45)
    ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
    for i in range(len(corr)):
        for j in range(len(corr)):
            ax.text(j, i, f'{corr.iat[i,j]:.2f}', ha='center', va='center')
    plt.colorbar(im, ax=ax)
ax.set_title('Corrélation des returns journaliers')
plt.tight_layout(); plt.show()

## 7. Indicateurs techniques (RSI, MACD, Bollinger)

In [ ]:
t  = next(iter(data))
d  = data[t].tail(250).copy()

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(d.index, d['Close'], color='#dbe8f4', label='Close', linewidth=1.2)
if 'BB_upper' in d:
    axes[0].plot(d.index, d['BB_upper'], color='#3b82f6', alpha=0.5, linewidth=0.8, label='BB upper')
    axes[0].plot(d.index, d['BB_lower'], color='#3b82f6', alpha=0.5, linewidth=0.8, label='BB lower')
    axes[0].fill_between(d.index, d['BB_lower'], d['BB_upper'], color='#3b82f6', alpha=0.08)
axes[0].set_title(f'{t} — Prix + Bollinger Bands'); axes[0].legend(fontsize=8)

axes[1].plot(d.index, d['RSI'], color='#a78bfa', linewidth=1)
axes[1].axhline(70, color='#f43f5e', linestyle='--', linewidth=0.8)
axes[1].axhline(30, color='#00d68f', linestyle='--', linewidth=0.8)
axes[1].set_ylim(0, 100); axes[1].set_title('RSI 14')

axes[2].plot(d.index, d['MACD'],        color='#00d68f', linewidth=1, label='MACD')
axes[2].plot(d.index, d['MACD_signal'], color='#f59e0b', linewidth=1, label='Signal')
if 'MACD_hist' in d:
    axes[2].bar(d.index, d['MACD_hist'], color=['#00d68f' if v>=0 else '#f43f5e' for v in d['MACD_hist']],
                alpha=0.5, width=1.0, label='Hist')
axes[2].axhline(0, color='#3d5269', linewidth=0.5)
axes[2].set_title('MACD 12/26/9'); axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()

## 8. Observations

- **Données propres** : `compute_indicators()` applique `dropna()` après calcul des indicateurs (warm-up de ~50 jours pour EMA_50). Aucun NaN ne doit subsister dans les CSV de `data/processed/`.
- **Cryptos vs actions** : les cryptos (BTC, ETH) tradent 7j/7 (~365 jours/an) alors que les actions sont ≈52% des jours calendaires (252 jours ouvrés). Ne pas comparer brutalement les volumes.
- **Volatilité** : les cryptos affichent une volatilité annualisée 2–3× supérieure aux actions. Le Sharpe proxy doit être interprété dans ce contexte.
- **Corrélations** : actions tech (AAPL, MSFT, GOOGL) typiquement >0.5 entre elles. BTC/ETH typiquement >0.7. Corrélation actions/cryptos généralement faible (<0.3).
- **Distribution des returns** : leptokurtique (kurtosis > 3) → fat tails. Le LSTM avec MSE peut sous-estimer les événements extrêmes.
- **Pour le LSTM** : la `Directional Accuracy` reste la métrique clé, le RMSE seul est trompeur sur des actifs volatils.